In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Wed Aug 14 14:06:14 PDT 2024


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [4]:
# Parameters
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

## Forecasted births and stillbirths

In [5]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate,
    "estimate",
    location.title(),
    years=2022,
).value

In [6]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)

In [7]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2022        2023        0.000315
                  15.0       20.0     2022        2023        0.008079
                  20.0       25.0     2022        2023        0.087454
                  25.0       30.0     2022        2023        0.110737
                                                                ...   
                  35.0       40.0     2022        2023        0.028718
                  40.0       45.0     2022        2023        0.009527
                  45.0       50.0     2022        2023        0.002772
                  50.0       55.0     2022        2023        0.000257
Name: value, Length: 9, dtype: float64

In [8]:
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
India     Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.028718
                  30.000000  35.000000  2030        2031        0.069523
                  20.000000  25.000000  2030        2031        0.087454
                  25.000000  30.000000  2030        2031        0.110737
Name: value, Length: 50, dtype: float64

In [9]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [10]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [11]:
pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031                 NaN
                  0.500000   1.000000    2030        2031                 NaN
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [12]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
India     Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [13]:
pop = pop.fillna(0)

In [14]:
n_births = (pop * asfr).sum()
n_births

19609719.34389394

In [15]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
    "estimate",
    location.title(),
    years=2022,
).value
sbr

location  year_start  year_end  parameter  
India     2022        2023      lower_value    0.016328
                                mean_value     0.016328
                                upper_value    0.016328
Name: value, dtype: float64

In [16]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
sbr

location  year_start  year_end
India     2022        2023        0.016328
Name: value, dtype: float64

In [17]:
sbr = sbr.values[0]

In [18]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

19.929907151540085

## Fertility (technically birth-and-stillbirth) disparities

In [19]:
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

1    4.828535e+06
2    4.265506e+06
3    3.829422e+06
4    3.610956e+06
5    3.075300e+06
dtype: float64

In [20]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

1    4.907375e+06
2    4.335154e+06
3    3.891949e+06
4    3.669916e+06
5    3.125514e+06
dtype: float64

In [21]:
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths for under-1 year olds
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate = ntd_deaths / n_births
10_000 * ntd_death_rate

2.1792101789211165

In [22]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

8.353639019197614

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [23]:
if location == "india":
    s_baseline_folate = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    s_baseline_folate = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    s_baseline_folate = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

In [24]:
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth

1    1
2    1
3    1
4    1
5    1
dtype: int64

In [25]:
s_ntd_death_rate = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate

1    2.17921
2    2.17921
3    2.17921
4    2.17921
5    2.17921
dtype: float64

In [26]:
s_ntd_death_count = s_ntd_death_rate * s_births_and_stillbirths_by_wealth
s_ntd_death_count

1    1069.420132
2     944.721078
3     848.137495
4     799.751860
5     681.115088
dtype: float64

In [27]:
s_ntd_death_count.sum(), ntd_deaths  # should be similar

(4343.145652958891, 4273.37)

In [28]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

1    3030.023707
2    2676.709722
3    2403.056236
4    2265.963603
5    1929.826082
dtype: float64

In [29]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

1    4099.443838
2    3621.430801
3    3251.193731
4    3065.715463
5    2610.941169
dtype: float64

In [30]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc


backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "daly")

1    1283.442766
2    1283.442766
3    1283.442766
4    1283.442766
5    1283.442766
dtype: float64

In [31]:
backcalc_rbc(
    s_ntd_death_or_stillbirth_count / s_births, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

1    941.439754
2    941.439754
3    941.439754
4    941.439754
5    941.439754
dtype: float64

In [32]:
if location == "india":
    assert vehicle == "rice"
    s_daily_vehicle = pd.Series(  # Zeb and Alix analysis of HCES
        {
            1: 213.570675,  # grams
            2: 175.027768,
            3: 163.699804,
            4: 163.363078,
            5: 127.874174,
        }
    )
elif location == "nigeria":
    assert vehicle == "bouillon"
    s_daily_vehicle = pd.Series(  # NFCMS 2021, Table 170. Usual intake of Bouillon (raw weight, grams) of women
        {
            1: 8.4,  # grams
            2: 8.0,
            3: 5.9,
            4: 4.9,
            5: 4.6,
        }
    )
elif location == "ethiopia":
    assert vehicle == "salt"
    s_daily_vehicle = (
        pd.Series(  # Dememoz Woldegebreal, personal communication of analysis
            {  # of 2013 Ethiopian National Food Consumption Survey (ENFCS)
                1: 7.5467,  # grams
                2: 6.3605,
                3: 6.5508,
                4: 6.5491,
                5: 6.5406,
            }
        )
        * 0.90
    )  # Saje et al 2024 assume 90% of total salt consumption comes from discretionary salt and manufactured food items (cites James et al 1987 )

In [33]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.125

In [34]:
intervention_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert intervention_concentration_mcg_per_gram.value.nunique() == 1
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.value.iloc[0]
)
intervention_concentration_mcg_per_gram

0.125

In [35]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"
eff_fort_intervention_path = f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()
df_eff_fort_intervention = pd.read_csv(eff_fort_intervention_path)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()

In [36]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,1.144883e+07
41,Female,15.0,20.0,2,1.300234e+07
42,Female,15.0,20.0,3,1.357840e+07
43,Female,15.0,20.0,4,1.349499e+07
...,...,...,...,...,...
71,Female,45.0,50.0,2,7.200663e+06
72,Female,45.0,50.0,3,7.650878e+06
73,Female,45.0,50.0,4,8.210394e+06
74,Female,45.0,50.0,5,8.760835e+06


In [37]:
if "sex" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[(df_eff_fort_baseline.sex == "Female")]

if "age_start" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[
        (df_eff_fort_baseline.age_start >= 15) & (df_eff_fort_baseline.age_end <= 50)
    ]

In [38]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [39]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population)
    return merged.groupby(["wealth_quintile"]).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [40]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

    wealth_quintile vehicle_name     sex  age_start_fort  age_end_fort  \
0                 1         rice  Female              15            30   
1                 1         rice  Female              15            30   
2                 1         rice  Female              15            30   
10                1         rice  Female              30            50   
..              ...          ...     ...             ...           ...   
66                5         rice  Female              30            50   
67                5         rice  Female              30            50   
68                5         rice  Female              30            50   
69                5         rice  Female              30            50   

    value_fort  index  age_start_pop  age_end_pop     value_pop  
0     0.340118     40           15.0         20.0  1.144883e+07  
1     0.340118     45           20.0         25.0  1.161927e+07  
2     0.340118     50           25.0         30.0  1.097286e+

wealth_quintile
1    0.353871
2    0.356231
3    0.318652
4    0.280922
5    0.157929
dtype: float64

In [41]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

       sex  age_start_fort  age_end_fort  wealth_quintile vehicle_name  \
0   Female              15            30                1         rice   
1   Female              15            30                1         rice   
2   Female              15            30                1         rice   
10  Female              30            50                1         rice   
..     ...             ...           ...              ...          ...   
66  Female              30            50                5         rice   
67  Female              30            50                5         rice   
68  Female              30            50                5         rice   
69  Female              30            50                5         rice   

    value_fort  index  age_start_pop  age_end_pop     value_pop  
0     0.490003     40           15.0         20.0  1.144883e+07  
1     0.490003     45           20.0         25.0  1.161927e+07  
2     0.490003     50           25.0         30.0  1.097286e+

wealth_quintile
1    0.496925
2    0.504283
3    0.490551
4    0.470491
5    0.415952
dtype: float64

In [42]:
RBC_baseline = backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "crider")
RBC_baseline

1    941.439754
2    941.439754
3    941.439754
4    941.439754
5    941.439754
dtype: float64

In [43]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [44]:
s_intervention_folate = (
    s_baseline_folate
    - (
        df_eff_fort_baseline
        * s_daily_vehicle
        * baseline_concentration_mcg_per_gram
        * fortification_mcg_to_dfe
    )
    + (
        df_eff_fort_intervention
        * s_daily_vehicle
        * intervention_concentration_mcg_per_gram
        * fortification_mcg_to_dfe
    )
)
s_intervention_folate

1    226.492346
2    225.506563
3    225.979700
4    226.580834
5    227.011336
dtype: float64

In [45]:
intevention_folate_pct_increase = (
    s_intervention_folate - s_baseline_folate
) / s_baseline_folate
intevention_folate_pct_increase

1    0.029511
2    0.025030
3    0.027180
4    0.029913
5    0.031870
dtype: float64

In [46]:
RBC_with_fort = RBC_baseline * (1 + ((6 / 10) * intevention_folate_pct_increase))
RBC_with_fort

1    958.109260
2    955.578200
3    956.793010
4    958.336459
5    959.441799
dtype: float64

In [47]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p


s_ntd_death_or_stillbirth_rate_with_fort = calc_ntd_pr(RBC_with_fort, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_with_fort

1    8.247460
2    8.284631
3    8.266757
4    8.244136
5    8.227997
dtype: float64

In [48]:
s_ntd_death_or_stillbirth_count_with_fort = (
    s_ntd_death_or_stillbirth_rate_with_fort * s_births
)
s_ntd_death_or_stillbirth_count_with_fort

1    3982.314525
2    3533.814714
3    3165.690407
4    2976.921664
5    2530.355767
dtype: float64

In [49]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_death_or_stillbirth_count.rename("value")
        .rename_axis("wealth_quintile")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_death_or_stillbirth_count_with_fort.rename("value")
        .rename_axis("wealth_quintile")
        .to_frame()
        .assign(entity="ntd", scenario=intervention_scenario)
        .set_index(["entity", "scenario"], append=True)
        .value,
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario    
1                ntd     baseline        4099.443838
2                ntd     baseline        3621.430801
3                ntd     baseline        3251.193731
4                ntd     baseline        3065.715463
                                            ...     
2                ntd     intervention    3533.814714
3                ntd     intervention    3165.690407
4                ntd     intervention    2976.921664
5                ntd     intervention    2530.355767
Name: value, Length: 10, dtype: float64

In [50]:
path = (
    f"./results/{location}/{vehicle}/{intervention_scenario}/ntd_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [51]:
# For calculating YLLs
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [52]:
# NOTE: Treating stillbirths as a death!
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [53]:
ylls_by_scenario = ntd_cases_by_scenario * yll_per_ntd
ylls_by_scenario

wealth_quintile  entity  scenario    
1                ntd     baseline        368777.931735
2                ntd     baseline        325776.815913
3                ntd     baseline        292471.014886
4                ntd     baseline        275785.753509
                                             ...      
2                ntd     intervention    317895.044530
3                ntd     intervention    284779.303424
4                ntd     intervention    267798.037330
5                ntd     intervention    227625.844664
Name: value, Length: 10, dtype: float64

In [54]:
path = f"./results/{location}/{vehicle}/{intervention_scenario}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)